# Two-Tower Controlled Experiments

目标：在不覆盖 `TT_V0_PV_UNIFORM` 的前提下，只改变 positive objective，训练并比较 `TT_V1_BUY_UNIFORM` 与 `TT_V2_MULTI_UNIFORM`。三个实验共用固定的 9,836-user / 302,016-item mapping、ID-only 双塔容量、uniform random negatives、3 epochs、batch size、optimizer、随机种子、训练步数和 warm-start purchase exact-retrieval evaluation。

本轮不做 in-batch negatives、hard negatives、FAISS IVF/HNSW、Ranking、candidate fusion 或架构搜索。

In [1]:
from pathlib import Path
from collections import defaultdict
import hashlib
import json
import math
import random
import time

import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader, RandomSampler, WeightedRandomSampler

cwd = Path.cwd().resolve()
PROJECT_ROOT = cwd.parent if cwd.name == 'notebooks' else cwd
DATA_DIR = PROJECT_ROOT / 'data' / 'processed' / 'dev'
ARTIFACT_ROOT = PROJECT_ROOT / 'artifacts' / 'two_tower'
EXPERIMENT_ROOT = ARTIFACT_ROOT / 'experiments'
EXPERIMENT_ROOT.mkdir(parents=True, exist_ok=True)
TRAIN_PATH = DATA_DIR / 'train.csv'
VALIDATION_GT_PATH = DATA_DIR / 'validation_ground_truth.csv'
V0_CHECKPOINT_PATH = ARTIFACT_ROOT / 'two_tower_best.pt'
USER_MAPPING_PATH = ARTIFACT_ROOT / 'user_mapping.csv'
ITEM_MAPPING_PATH = ARTIFACT_ROOT / 'item_mapping.csv'
MAPPING_METADATA_PATH = ARTIFACT_ROOT / 'mapping_metadata.json'

train_df = pd.read_csv(TRAIN_PATH)
validation_gt = pd.read_csv(VALIDATION_GT_PATH)
user_mapping = pd.read_csv(USER_MAPPING_PATH).sort_values('user_idx')
item_mapping = pd.read_csv(ITEM_MAPPING_PATH).sort_values('item_idx')
mapping_metadata = json.loads(MAPPING_METADATA_PATH.read_text(encoding='utf-8'))
user2idx = dict(zip(user_mapping.user_id.astype(int), user_mapping.user_idx.astype(int)))
item2idx = dict(zip(item_mapping.item_id.astype(int), item_mapping.item_idx.astype(int)))
idx2item = dict(zip(item_mapping.item_idx.astype(int), item_mapping.item_id.astype(int)))
candidate_item_ids = item_mapping.item_id.to_numpy(dtype=np.int64)
NUM_USERS, NUM_ITEMS = len(user_mapping), len(item_mapping)
assert (NUM_USERS, NUM_ITEMS) == (9_836, 302_016)
assert mapping_metadata['num_users'] == NUM_USERS and mapping_metadata['num_items'] == NUM_ITEMS

user_seen = validation_gt.user_id.isin(user2idx)
item_seen = validation_gt.item_id.isin(item2idx)
warm_gt_df = validation_gt.loc[user_seen & item_seen, ['user_id', 'item_id']].drop_duplicates().copy()
warm_ground_truth = warm_gt_df.groupby('user_id').item_id.apply(set).to_dict()
warm_user_ids = np.array(sorted(warm_ground_truth), dtype=np.int64)
warm_user_indices = np.array([user2idx[int(uid)] for uid in warm_user_ids], dtype=np.int64)
validation_user_indices = warm_gt_df.user_id.map(user2idx).to_numpy(dtype=np.int64).copy()
validation_item_indices = warm_gt_df.item_id.map(item2idx).to_numpy(dtype=np.int64).copy()

def seed_everything(seed):
    random.seed(seed); np.random.seed(seed); torch.manual_seed(seed)

class UserTower(nn.Module):
    def __init__(self, num_users, embedding_dim=64, hidden_dim=128, output_dim=64):
        super().__init__(); self.user_embedding = nn.Embedding(num_users, embedding_dim)
        self.mlp = nn.Sequential(nn.Linear(embedding_dim, hidden_dim), nn.ReLU(), nn.Linear(hidden_dim, output_dim))
    def forward(self, ids): return F.normalize(self.mlp(self.user_embedding(ids)), p=2, dim=-1)

class ItemTower(nn.Module):
    def __init__(self, num_items, embedding_dim=64, hidden_dim=128, output_dim=64):
        super().__init__(); self.item_embedding = nn.Embedding(num_items, embedding_dim)
        self.mlp = nn.Sequential(nn.Linear(embedding_dim, hidden_dim), nn.ReLU(), nn.Linear(hidden_dim, output_dim))
    def forward(self, ids): return F.normalize(self.mlp(self.item_embedding(ids)), p=2, dim=-1)

class TwoTowerModel(nn.Module):
    def __init__(self, num_users=NUM_USERS, num_items=NUM_ITEMS, embedding_dim=64, hidden_dim=128, output_dim=64, temperature=0.1):
        super().__init__(); self.user_tower = UserTower(num_users, embedding_dim, hidden_dim, output_dim)
        self.item_tower = ItemTower(num_items, embedding_dim, hidden_dim, output_dim); self.temperature = temperature
    def encode_users(self, ids): return self.user_tower(ids)
    def encode_items(self, ids): return self.item_tower(ids)
    def forward(self, users, positives, negatives):
        uv, pv, nv = self.encode_users(users), self.encode_items(positives), self.encode_items(negatives)
        return (uv * pv).sum(-1) / self.temperature, (uv.unsqueeze(1) * nv).sum(-1) / self.temperature

def sampled_bce_loss(positive_logits, negative_logits):
    pos = F.binary_cross_entropy_with_logits(positive_logits, torch.ones_like(positive_logits))
    neg = F.binary_cross_entropy_with_logits(negative_logits, torch.zeros_like(negative_logits))
    return (pos + neg) / 2

class UniformNegativeDataset(Dataset):
    def __init__(self, pairs, positive_by_user, num_items, num_negatives, seed):
        self.pairs = np.asarray(pairs, dtype=np.int64); self.positive_by_user = positive_by_user
        self.num_items = num_items; self.num_negatives = num_negatives; self.rng = np.random.default_rng(seed)
    def __len__(self): return len(self.pairs)
    def __getitem__(self, index):
        user_idx, positive_idx = self.pairs[index]; excluded = self.positive_by_user[int(user_idx)]; negatives = []
        while len(negatives) < self.num_negatives:
            candidate = int(self.rng.integers(0, self.num_items))
            if candidate != positive_idx and candidate not in excluded and candidate not in negatives: negatives.append(candidate)
        return {'user': torch.tensor(user_idx), 'positive_item': torch.tensor(positive_idx), 'negative_items': torch.tensor(negatives)}

def build_positive_sets(pairs):
    result = defaultdict(set)
    for user_idx, item_idx in np.asarray(pairs, dtype=np.int64): result[int(user_idx)].add(int(item_idx))
    return result

known_positive_by_user = defaultdict(set)
mapped_known = train_df.loc[train_df.user_id.isin(user2idx) & train_df.item_id.isin(item2idx), ['user_id', 'item_id']].drop_duplicates()
for row in mapped_known.itertuples(index=False): known_positive_by_user[user2idx[int(row.user_id)]].add(item2idx[int(row.item_id)])
for user_idx, item_idx in zip(validation_user_indices, validation_item_indices): known_positive_by_user[int(user_idx)].add(int(item_idx))
fixed_rng = np.random.default_rng(2026)
fixed_validation_negatives = np.empty((len(warm_gt_df), 4), dtype=np.int64)
for row, (user_idx, positive_idx) in enumerate(zip(validation_user_indices, validation_item_indices)):
    negatives = []
    while len(negatives) < 4:
        candidate = int(fixed_rng.integers(0, NUM_ITEMS))
        if candidate not in known_positive_by_user[int(user_idx)] and candidate not in negatives: negatives.append(candidate)
    fixed_validation_negatives[row] = negatives
assert all(len(set(row)) == 4 for row in fixed_validation_negatives)
assert all(int(neg) not in known_positive_by_user[int(user)] for user, row in zip(validation_user_indices, fixed_validation_negatives) for neg in row)

@torch.inference_mode()
def deterministic_validation_loss(model, batch_size=1024):
    losses = []
    for start in range(0, len(warm_gt_df), batch_size):
        end = min(start + batch_size, len(warm_gt_df))
        users = torch.from_numpy(validation_user_indices[start:end])
        positives = torch.from_numpy(validation_item_indices[start:end])
        negatives = torch.from_numpy(fixed_validation_negatives[start:end])
        pos_logits, neg_logits = model(users, positives, negatives)
        losses.append((sampled_bce_loss(pos_logits, neg_logits).item(), end - start))
    return sum(loss * size for loss, size in losses) / len(warm_gt_df)

@torch.inference_mode()
def encode_all(model, batch_size=8192):
    users = [model.encode_users(torch.arange(s, min(s + batch_size, NUM_USERS))).numpy() for s in range(0, NUM_USERS, batch_size)]
    items = [model.encode_items(torch.arange(s, min(s + batch_size, NUM_ITEMS))).numpy() for s in range(0, NUM_ITEMS, batch_size)]
    return np.concatenate(users).astype('float32'), np.concatenate(items).astype('float32')

def exact_chunked_topk(user_embeddings, item_embeddings, user_indices, top_k=50, user_batch_size=64, item_chunk_size=50000):
    item_tensor = torch.from_numpy(item_embeddings); result_indices = []
    for user_start in range(0, len(user_indices), user_batch_size):
        batch_indices = user_indices[user_start:user_start + user_batch_size]
        user_tensor = torch.from_numpy(user_embeddings[batch_indices]); batch_size = len(batch_indices)
        best_scores = torch.full((batch_size, top_k), -torch.inf); best_indices = torch.full((batch_size, top_k), -1, dtype=torch.long)
        for item_start in range(0, NUM_ITEMS, item_chunk_size):
            item_end = min(item_start + item_chunk_size, NUM_ITEMS); scores = user_tensor @ item_tensor[item_start:item_end].T
            local_scores, local_indices = torch.topk(scores, min(top_k, item_end - item_start), dim=1); local_indices += item_start
            merged_scores = torch.cat([best_scores, local_scores], dim=1); merged_indices = torch.cat([best_indices, local_indices], dim=1)
            best_scores, positions = torch.topk(merged_scores, top_k, dim=1); best_indices = torch.gather(merged_indices, 1, positions)
        result_indices.append(best_indices.numpy())
    return np.vstack(result_indices)

def recall_at_k(recs, gt, k): return len(set(recs[:k]) & set(gt)) / len(gt) if gt else 0.0
def hitrate_at_k(recs, gt, k): return float(bool(set(recs[:k]) & set(gt))) if gt else 0.0
def ndcg_at_k(recs, gt, k):
    if not gt: return 0.0
    dcg = sum(1 / math.log2(rank + 1) for rank, item in enumerate(recs[:k], 1) if item in gt)
    idcg = sum(1 / math.log2(rank + 1) for rank in range(1, min(len(gt), k) + 1))
    return dcg / idcg if idcg else 0.0

def evaluate_recommendations(recommendations, ks=(10, 20, 50)):
    rows = []
    for k in ks:
        rows.append({'K': k, 'Recall': np.mean([recall_at_k(recommendations[u], gt, k) for u, gt in warm_ground_truth.items()]), 'HitRate': np.mean([hitrate_at_k(recommendations[u], gt, k) for u, gt in warm_ground_truth.items()]), 'NDCG': np.mean([ndcg_at_k(recommendations[u], gt, k) for u, gt in warm_ground_truth.items()])})
    return pd.DataFrame(rows)

def evaluate_model(model):
    user_embeddings, item_embeddings = encode_all(model); top_indices = exact_chunked_topk(user_embeddings, item_embeddings, warm_user_indices)
    recommendations = {int(uid): candidate_item_ids[top_indices[row]].astype(int).tolist() for row, uid in enumerate(warm_user_ids)}
    metrics = evaluate_recommendations(recommendations)
    top1 = pd.Series([items[0] for items in recommendations.values()]).value_counts(); catalog = {item for items in recommendations.values() for item in items}
    concentration = {'unique_top1': int(len(top1)), 'dominant_top1_item': int(top1.index[0]), 'dominant_top1_share': float(top1.iloc[0] / len(recommendations)), 'unique_top50': int(len(catalog)), 'catalog_coverage_at_50': float(len(catalog) / NUM_ITEMS)}
    return metrics, recommendations, concentration

def json_write(path, payload): path.write_text(json.dumps(payload, indent=2, ensure_ascii=False), encoding='utf-8')

print('Project root:', PROJECT_ROOT)
print('Shared vocabulary:', NUM_USERS, 'users /', NUM_ITEMS, 'items')
print('Warm purchase evaluation:', len(warm_ground_truth), 'users /', len(warm_gt_df), 'interactions')
print('Fixed validation negative shape:', fixed_validation_negatives.shape)
print('Validation negatives exclude all known train interactions and validation GT: OK')

Project root: <LOCAL_PROJECT_PATH>
Shared vocabulary: 9836 users / 302016 items
Warm purchase evaluation: 1170 users / 1496 interactions
Fixed validation negative shape: (1496, 4)
Validation negatives exclude all known train interactions and validation GT: OK


## Behavior statistics and weighting rationale

先统计每种行为的行数、unique user-item pairs，以及这些历史 pair 与未来 validation purchase 的重叠率。Multi-behavior 不把四类行为视为完全相同：使用相对 PV purchase-overlap lift 的平方根作为 sampling weight，以保留方向又避免按原始 lift 过度放大。每个 user-item 只保留一次，并使用其最强行为权重。

In [2]:
gt_pair_set = set(map(tuple, validation_gt[['user_id', 'item_id']].to_numpy()))
behavior_rows = []
for behavior in ['pv', 'fav', 'cart', 'buy']:
    rows = train_df.loc[train_df.behavior_type.eq(behavior)]
    pairs = rows[['user_id', 'item_id']].drop_duplicates()
    overlap = sum(tuple(pair) in gt_pair_set for pair in pairs.to_numpy())
    behavior_rows.append({'behavior': behavior, 'rows': len(rows), 'unique_pairs': len(pairs), 'users': pairs.user_id.nunique(), 'items': pairs.item_id.nunique(), 'future_purchase_overlap': overlap, 'future_purchase_rate': overlap / len(pairs)})
behavior_stats = pd.DataFrame(behavior_rows)
pv_rate = float(behavior_stats.loc[behavior_stats.behavior.eq('pv'), 'future_purchase_rate'].iloc[0])
behavior_stats['lift_vs_pv'] = behavior_stats.future_purchase_rate / pv_rate
behavior_stats['sampling_weight'] = np.sqrt(behavior_stats.lift_vs_pv)
BEHAVIOR_WEIGHTS = dict(zip(behavior_stats.behavior, behavior_stats.sampling_weight.astype(float)))
behavior_stats.to_csv(EXPERIMENT_ROOT / 'behavior_statistics.csv', index=False)
display(behavior_stats)
print('Chosen multi-behavior sampling weights:', {key: round(value, 4) for key, value in BEHAVIOR_WEIGHTS.items()})

,behavior,rows,unique_pairs,users,items,future_purchase_overlap,future_purchase_rate,lift_vs_pv,sampling_weight
0,pv,649470,521287,9836,302016,457,0.000877,1.000000,1.000000
1,fav,20367,20283,3417,18943,69,0.003402,3.880410,1.969876
2,cart,40241,39304,6715,34888,372,0.009465,10.796100,3.285742
3,buy,14807,14212,5696,13145,72,0.005066,5.778804,2.403914


Chosen multi-behavior sampling weights: {'pv': 1.0, 'fav': 1.9699, 'cart': 3.2857, 'buy': 2.4039}


## Controlled configuration

V1/V2 都使用与 V0 相同的初始化 seed、64-d embedding、128 hidden、64 output、temperature 0.1、Adam 1e-3、batch 1024、4 个 uniform negatives、3 epochs。每个 epoch 固定抽取 521,287 个 positives，即与 V0 每个 epoch 的样本数一致。Buy-only 因正样本很少而采用有放回均匀采样；Multi-behavior 使用上述权重进行有放回采样。核心 model selection 使用固定 warm purchase exact Recall@50，deterministic sampled loss 只作诊断。

In [3]:
BASE_CONFIG = {'seed': 42, 'epochs': 3, 'epoch_samples': 521_287, 'batch_size': 1024, 'num_negatives': 4, 'learning_rate': 0.001, 'embedding_dim': 64, 'hidden_dim': 128, 'output_dim': 64, 'temperature': 0.1, 'candidate_count': NUM_ITEMS, 'evaluation': 'warm validation purchase exact retrieval; no history filtering'}
mapped_train = train_df.loc[train_df.user_id.isin(user2idx) & train_df.item_id.isin(item2idx), ['user_id', 'item_id', 'behavior_type']].copy()
mapped_train['user_idx'] = mapped_train.user_id.map(user2idx).astype(int)
mapped_train['item_idx'] = mapped_train.item_id.map(item2idx).astype(int)

buy_pairs_df = mapped_train.loc[mapped_train.behavior_type.eq('buy'), ['user_idx', 'item_idx']].drop_duplicates().reset_index(drop=True)
buy_pairs = buy_pairs_df[['user_idx', 'item_idx']].to_numpy(dtype=np.int64)
buy_positive_sets = build_positive_sets(buy_pairs)

mapped_train['behavior_weight'] = mapped_train.behavior_type.map(BEHAVIOR_WEIGHTS).astype(float)
multi_pairs_df = mapped_train.groupby(['user_idx', 'item_idx'], as_index=False).agg(sampling_weight=('behavior_weight', 'max'))
multi_pairs = multi_pairs_df[['user_idx', 'item_idx']].to_numpy(dtype=np.int64)
multi_weights = multi_pairs_df.sampling_weight.to_numpy(dtype=np.float64)
multi_positive_sets = build_positive_sets(multi_pairs)

buy_eval_users_with_training_positive = sum(user2idx[int(uid)] in buy_positive_sets for uid in warm_user_ids)
print('Buy-only mapped positives:', len(buy_pairs), 'pairs /', buy_pairs_df.user_idx.nunique(), 'users /', buy_pairs_df.item_idx.nunique(), 'items')
print('Warm eval users with at least one train buy:', buy_eval_users_with_training_positive, '/', len(warm_user_ids))
print('Buy-only mean resamples per positive per epoch:', BASE_CONFIG['epoch_samples'] / len(buy_pairs))
print('Multi-behavior mapped positives:', len(multi_pairs), 'pairs /', multi_pairs_df.user_idx.nunique(), 'users /', multi_pairs_df.item_idx.nunique(), 'items')
print('Epoch samples/batches for both experiments:', BASE_CONFIG['epoch_samples'], '/', math.ceil(BASE_CONFIG['epoch_samples'] / BASE_CONFIG['batch_size']))

Buy-only mapped positives: 11844 pairs / 5141 users / 10819 items
Warm eval users with at least one train buy: 712 / 1170
Buy-only mean resamples per positive per epoch: 44.01274907125971
Multi-behavior mapped positives: 541958 pairs / 9836 users / 302016 items
Epoch samples/batches for both experiments: 521287 / 510


## Training and persistence

每个实验重置相同随机种子并从相同初始化分布开始。每个 epoch 输出 train loss、固定 purchase sampled loss、exact Recall@50/NDCG@50。最佳模型按 Recall@50、再按 NDCG@50 选择，并保存到独立目录；checkpoint 内嵌完整 config 和 mapping SHA/reference，不覆盖 V0。

In [4]:
def train_experiment(name, positive_definition, pairs, positive_sets, sampling_weights=None, behavior_weights=None):
    config = dict(BASE_CONFIG)
    config.update({'experiment_name': name, 'positive_definition': positive_definition, 'negative_strategy': 'uniform random; 4 distinct negatives; exclude objective-positive items', 'sampler': 'weighted replacement' if sampling_weights is not None else 'uniform replacement', 'positive_pair_count': int(len(pairs)), 'positive_user_count': int(len(positive_sets)), 'behavior_weights': behavior_weights, 'mapping': {'user_mapping': '../../user_mapping.csv', 'item_mapping': '../../item_mapping.csv', 'train_csv_sha256': mapping_metadata['train_csv_sha256']}})
    output_dir = EXPERIMENT_ROOT / name; output_dir.mkdir(parents=True, exist_ok=True)
    checkpoint_path = output_dir / 'checkpoint_best.pt'; json_write(output_dir / 'config.json', config)
    seed_everything(config['seed']); model = TwoTowerModel(); optimizer = torch.optim.Adam(model.parameters(), lr=config['learning_rate'])
    dataset = UniformNegativeDataset(pairs, positive_sets, NUM_ITEMS, config['num_negatives'], config['seed'])
    generator = torch.Generator().manual_seed(config['seed'])
    if sampling_weights is None:
        sampler = RandomSampler(dataset, replacement=True, num_samples=config['epoch_samples'], generator=generator)
    else:
        sampler = WeightedRandomSampler(torch.as_tensor(sampling_weights, dtype=torch.double), num_samples=config['epoch_samples'], replacement=True, generator=generator)
    loader = DataLoader(dataset, batch_size=config['batch_size'], sampler=sampler, num_workers=0)
    history = []; best_key = (-1.0, -1.0)
    for epoch in range(1, config['epochs'] + 1):
        model.train(); total_loss = 0.0; started = time.perf_counter()
        for batch_id, batch in enumerate(loader, 1):
            optimizer.zero_grad(); pos_logits, neg_logits = model(batch['user'].long(), batch['positive_item'].long(), batch['negative_items'].long())
            loss = sampled_bce_loss(pos_logits, neg_logits); loss.backward(); optimizer.step(); total_loss += loss.item()
            if batch_id % 100 == 0: print(f'{name} epoch {epoch} batch {batch_id}/{len(loader)}', flush=True)
        train_loss = total_loss / len(loader); model.eval(); fixed_loss = deterministic_validation_loss(model)
        metrics, _, concentration = evaluate_model(model); r50 = float(metrics.loc[metrics.K.eq(50), 'Recall'].iloc[0]); n50 = float(metrics.loc[metrics.K.eq(50), 'NDCG'].iloc[0])
        row = {'epoch': epoch, 'train_loss': train_loss, 'fixed_purchase_sampled_loss': fixed_loss, 'Recall@50': r50, 'NDCG@50': n50, 'seconds': time.perf_counter() - started, **concentration}; history.append(row)
        print(f"{name} epoch {epoch}: train={train_loss:.6f} fixed_val={fixed_loss:.6f} R@50={r50:.6f} N@50={n50:.6f} seconds={row['seconds']:.1f}", flush=True)
        if (r50, n50) > best_key:
            best_key = (r50, n50)
            torch.save({'model_state_dict': model.state_dict(), 'num_users': NUM_USERS, 'num_items': NUM_ITEMS, 'embedding_dim': 64, 'hidden_dim': 128, 'output_dim': 64, 'temperature': 0.1, 'best_epoch': epoch, 'best_recall_at_50': r50, 'best_ndcg_at_50': n50, 'config': config, 'mapping': {'user_mapping': '../../user_mapping.csv', 'item_mapping': '../../item_mapping.csv', 'train_csv_sha256': mapping_metadata['train_csv_sha256']}}, checkpoint_path)
    saved = torch.load(checkpoint_path, map_location='cpu', weights_only=True); model.load_state_dict(saved['model_state_dict'], strict=True); model.eval()
    final_metrics, recommendations, final_concentration = evaluate_model(model); final_metrics.insert(0, 'Model', name)
    pd.DataFrame(history).to_csv(output_dir / 'training_history.csv', index=False); final_metrics.to_csv(output_dir / 'metrics.csv', index=False); json_write(output_dir / 'concentration.json', final_concentration)
    sample_ids = np.random.default_rng(42).choice(warm_user_ids, 5, replace=False)
    pd.DataFrame([{'user_id': int(uid), 'retrieved_top10': recommendations[int(uid)][:10], 'actual_purchase_gt': sorted(warm_ground_truth[int(uid)])} for uid in sample_ids]).to_csv(output_dir / 'sample_recommendations.csv', index=False)
    return {'name': name, 'config': config, 'model': model, 'history': pd.DataFrame(history), 'metrics': final_metrics, 'recommendations': recommendations, 'concentration': final_concentration, 'checkpoint': checkpoint_path}

print('Training helper ready. V0 checkpoint will only be read, never written.')

Training helper ready. V0 checkpoint will only be read, never written.


## Experiment 1 — TT_V1_BUY_UNIFORM

Positive 只使用 train-period buy user-item pairs，并限制在共享 V0 vocabulary 内。由于可映射 pair 很少，通过有放回采样保持与 V0 相同的每-epoch optimizer steps；这是控制训练预算所需的最小调整，也会带来重复曝光和过拟合风险。

In [5]:
buy_result = train_experiment('TT_V1_BUY_UNIFORM', "train-period behavior_type == 'buy'; unique mapped user-item pairs", buy_pairs, buy_positive_sets)
display(buy_result['history'])
display(buy_result['metrics'])
print('Concentration:', buy_result['concentration'])
print('Checkpoint:', buy_result['checkpoint'])

TT_V1_BUY_UNIFORM epoch 1 batch 100/510


TT_V1_BUY_UNIFORM epoch 1 batch 200/510


TT_V1_BUY_UNIFORM epoch 1 batch 300/510


TT_V1_BUY_UNIFORM epoch 1 batch 400/510


TT_V1_BUY_UNIFORM epoch 1 batch 500/510


...

TT_V1_BUY_UNIFORM epoch 3 batch 500/510


TT_V1_BUY_UNIFORM epoch 3: train=0.040999 fixed_val=2.208330 R@50=0.011966 N@50=0.003453 seconds=60.3


,epoch,train_loss,fixed_purchase_sampled_loss,Recall@50,NDCG@50,seconds,unique_top1,dominant_top1_item,dominant_top1_share,unique_top50,catalog_coverage_at_50
0,1,0.398223,1.362624,0.003419,0.000764,61.212656,631,4737932,0.015385,9734,0.032230
1,2,0.098541,2.001213,0.008974,0.002751,60.978938,397,3749656,0.030769,3810,0.012615
2,3,0.040999,2.208330,0.011966,0.003453,60.268247,393,2560262,0.053846,3934,0.013026


,Model,K,Recall,HitRate,NDCG
0,TT_V1_BUY_UNIFORM,10,0.004274,0.005128,0.001903
1,TT_V1_BUY_UNIFORM,20,0.004274,0.005128,0.001903
2,TT_V1_BUY_UNIFORM,50,0.011966,0.012821,0.003453


Concentration: {'unique_top1': 393, 'dominant_top1_item': 2560262, 'dominant_top1_share': 0.05384615384615385, 'unique_top50': 3934, 'catalog_coverage_at_50': 0.013025799957618139}
Checkpoint: <LOCAL_PROJECT_PATH>\artifacts\two_tower\experiments\TT_V1_BUY_UNIFORM\checkpoint_best.pt


## Experiment 2 — TT_V2_MULTI_UNIFORM

Positive 使用映射范围内所有 pv/fav/cart/buy user-item pairs，但不是等权。对同一 pair 取其最强历史行为的 data-derived square-root-lift weight，再用 WeightedRandomSampler 固定抽取 521,287 个 positives/epoch。Negative 仍为 uniform random。

In [6]:
multi_result = train_experiment('TT_V2_MULTI_UNIFORM', 'all mapped train behaviors; one user-item pair weighted by strongest observed behavior using sqrt future-purchase lift vs PV', multi_pairs, multi_positive_sets, multi_weights, {key: float(value) for key, value in BEHAVIOR_WEIGHTS.items()})
display(multi_result['history'])
display(multi_result['metrics'])
print('Concentration:', multi_result['concentration'])
print('Checkpoint:', multi_result['checkpoint'])

TT_V2_MULTI_UNIFORM epoch 1 batch 100/510


TT_V2_MULTI_UNIFORM epoch 1 batch 200/510


TT_V2_MULTI_UNIFORM epoch 1 batch 300/510


TT_V2_MULTI_UNIFORM epoch 1 batch 400/510


TT_V2_MULTI_UNIFORM epoch 1 batch 500/510


...

TT_V2_MULTI_UNIFORM epoch 3 batch 500/510


TT_V2_MULTI_UNIFORM epoch 3: train=0.652971 fixed_val=0.634544 R@50=0.005918 N@50=0.001569 seconds=61.3


,epoch,train_loss,fixed_purchase_sampled_loss,Recall@50,NDCG@50,seconds,unique_top1,dominant_top1_item,dominant_top1_share,unique_top50,catalog_coverage_at_50
0,1,0.695109,0.687192,0.000078,0.000076,61.205317,93,3845720,0.580342,2911,0.009639
1,2,0.678471,0.662867,0.007058,0.001677,62.068630,11,3845720,0.811966,688,0.002278
2,3,0.652971,0.634544,0.005918,0.001569,61.266633,28,3371523,0.478632,550,0.001821


,Model,K,Recall,HitRate,NDCG
0,TT_V2_MULTI_UNIFORM,10,0.000078,0.000855,0.000054
1,TT_V2_MULTI_UNIFORM,20,0.003069,0.005128,0.000858
2,TT_V2_MULTI_UNIFORM,50,0.007058,0.010256,0.001677


Concentration: {'unique_top1': 11, 'dominant_top1_item': 3845720, 'dominant_top1_share': 0.811965811965812, 'unique_top50': 688, 'catalog_coverage_at_50': 0.0022780250052977327}
Checkpoint: <LOCAL_PROJECT_PATH>\artifacts\two_tower\experiments\TT_V2_MULTI_UNIFORM\checkpoint_best.pt


## Controlled comparison

重新只读加载 V0 checkpoint，并让 V0/V1/V2 使用完全相同的 warm purchase GT、candidate catalog、history policy、exact Top-K 和 metric functions。除 Recall/HitRate/NDCG 外，同时报告 catalog coverage 与 Top-1/Top-50 concentration。

In [7]:
v0_checkpoint = torch.load(V0_CHECKPOINT_PATH, map_location='cpu', weights_only=True)
assert (v0_checkpoint['num_users'], v0_checkpoint['num_items']) == (NUM_USERS, NUM_ITEMS)
v0_model = TwoTowerModel(); v0_model.load_state_dict(v0_checkpoint['model_state_dict'], strict=True); v0_model.eval()
v0_metrics, v0_recommendations, v0_concentration = evaluate_model(v0_model); v0_metrics.insert(0, 'Model', 'TT_V0_PV_UNIFORM')
comparison_metrics = pd.concat([v0_metrics, buy_result['metrics'], multi_result['metrics']], ignore_index=True)
comparison_concentration = pd.DataFrame([{'Model': 'TT_V0_PV_UNIFORM', **v0_concentration}, {'Model': buy_result['name'], **buy_result['concentration']}, {'Model': multi_result['name'], **multi_result['concentration']}])
coverage = pd.DataFrame([{'GT_users': validation_gt.user_id.nunique(), 'GT_interactions': len(validation_gt), 'unseen_users': validation_gt.loc[~user_seen, 'user_id'].nunique(), 'unseen_target_interactions': int((~item_seen).sum()), 'warm_users': len(warm_ground_truth), 'warm_interactions': len(warm_gt_df), 'warm_user_coverage': len(warm_ground_truth) / validation_gt.user_id.nunique(), 'warm_interaction_coverage': len(warm_gt_df) / len(validation_gt)}])
comparison_metrics.to_csv(EXPERIMENT_ROOT / 'controlled_metrics.csv', index=False); comparison_concentration.to_csv(EXPERIMENT_ROOT / 'controlled_concentration.csv', index=False); coverage.to_csv(EXPERIMENT_ROOT / 'controlled_coverage.csv', index=False)
v0_dir = EXPERIMENT_ROOT / 'TT_V0_PV_UNIFORM'; v0_dir.mkdir(parents=True, exist_ok=True)
v0_record = {**BASE_CONFIG, 'experiment_name': 'TT_V0_PV_UNIFORM', 'positive_definition': "train-period behavior_type == 'pv'; unique user-item pairs", 'negative_strategy': 'original V0 uniform random sampler excluding train-PV positives', 'checkpoint': '../../two_tower_best.pt', 'original_train_loss': [0.698493, 0.690275, 0.679218], 'original_validation_loss': [0.692557, 0.682466, 0.660640], 'mapping': {'user_mapping': '../../user_mapping.csv', 'item_mapping': '../../item_mapping.csv', 'train_csv_sha256': mapping_metadata['train_csv_sha256']}}
json_write(v0_dir / 'experiment_record.json', v0_record); v0_metrics.to_csv(v0_dir / 'metrics.csv', index=False); json_write(v0_dir / 'concentration.json', v0_concentration)
display(behavior_stats)
display(coverage)
display(comparison_metrics)
display(comparison_concentration)

,behavior,rows,unique_pairs,users,items,future_purchase_overlap,future_purchase_rate,lift_vs_pv,sampling_weight
0,pv,649470,521287,9836,302016,457,0.000877,1.000000,1.000000
1,fav,20367,20283,3417,18943,69,0.003402,3.880410,1.969876
2,cart,40241,39304,6715,34888,372,0.009465,10.796100,3.285742
3,buy,14807,14212,5696,13145,72,0.005066,5.778804,2.403914


,GT_users,GT_interactions,unseen_users,unseen_target_interactions,warm_users,warm_interactions,warm_user_coverage,warm_interaction_coverage
0,1760,2540,16,1032,1170,1496,0.664773,0.588976


,Model,K,Recall,HitRate,NDCG
0,TT_V0_PV_UNIFORM,10,0.002564,0.002564,0.000920
1,TT_V0_PV_UNIFORM,20,0.002991,0.003419,0.001066
2,TT_V0_PV_UNIFORM,50,0.003846,0.004274,0.001221
3,TT_V1_BUY_UNIFORM,10,0.004274,0.005128,0.001903
4,TT_V1_BUY_UNIFORM,20,0.004274,0.005128,0.001903
5,TT_V1_BUY_UNIFORM,50,0.011966,0.012821,0.003453
6,TT_V2_MULTI_UNIFORM,10,0.000078,0.000855,0.000054
7,TT_V2_MULTI_UNIFORM,20,0.003069,0.005128,0.000858
8,TT_V2_MULTI_UNIFORM,50,0.007058,0.010256,0.001677


,Model,unique_top1,dominant_top1_item,dominant_top1_share,unique_top50,catalog_coverage_at_50
0,TT_V0_PV_UNIFORM,2,3845720,0.919658,301,0.000997
1,TT_V1_BUY_UNIFORM,393,2560262,0.053846,3934,0.013026
2,TT_V2_MULTI_UNIFORM,11,3845720,0.811966,688,0.002278


## Decision boundary

结果：Buy-only 最佳 epoch 3，Recall@50=0.011966；weighted multi-behavior 最佳 epoch 2，Recall@50=0.007058；两者都高于 V0 的 0.003846，且 Buy-only 明显降低 recommendation concentration。结果支持 positive-objective mismatch hypothesis，但单 seed、V1 稀疏 positives 和有放回过采样意味着这还不是单因素因果证明。Notebook 在对比后停止，不进入 Ranking。

In [8]:
r50_table = comparison_metrics.loc[comparison_metrics.K.eq(50), ['Model', 'Recall', 'HitRate', 'NDCG']].sort_values('Recall', ascending=False).reset_index(drop=True)
best_model = str(r50_table.iloc[0].Model); v0_r50 = float(r50_table.loc[r50_table.Model.eq('TT_V0_PV_UNIFORM'), 'Recall'].iloc[0]); buy_r50 = float(r50_table.loc[r50_table.Model.eq('TT_V1_BUY_UNIFORM'), 'Recall'].iloc[0]); multi_r50 = float(r50_table.loc[r50_table.Model.eq('TT_V2_MULTI_UNIFORM'), 'Recall'].iloc[0])
print('Recall@50 ranking:'); display(r50_table)
print('Best objective:', best_model)
print('Buy-only delta vs V0:', buy_r50 - v0_r50)
print('Multi-behavior delta vs V0:', multi_r50 - v0_r50)
print('Objective-mismatch hypothesis supported by metric improvement:', max(buy_r50, multi_r50) > v0_r50)
print('NEXT_RECOMMENDED_STEP: inspect stability and concentration of the winning objective, then run one minimal confirmatory seed before any Ranking work.')

Recall@50 ranking:


,Model,Recall,HitRate,NDCG
0,TT_V1_BUY_UNIFORM,0.011966,0.012821,0.003453
1,TT_V2_MULTI_UNIFORM,0.007058,0.010256,0.001677
2,TT_V0_PV_UNIFORM,0.003846,0.004274,0.001221


Best objective: TT_V1_BUY_UNIFORM
Buy-only delta vs V0: 0.00811965811965812
Multi-behavior delta vs V0: 0.003211603211603212
Objective-mismatch hypothesis supported by metric improvement: True
NEXT_RECOMMENDED_STEP: inspect stability and concentration of the winning objective, then run one minimal confirmatory seed before any Ranking work.
